### save_model()

Serialize a model's public parameters to a JSON file.

This cell verifies the `save_model` function mathematically against its cleartext counterpart, using a dynamically generated input set that covers positive, negative, zero, and array edge cases while respecting the `[-7, 7]` FHE RAM constraints.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.serialization import save_model

def test_save_model(model, path):
    import numpy as np
    res = save_model(model, path)
    return np.array(res) if isinstance(res, list) else res

compiler = fhe.Compiler(test_save_model, {'model': 'encrypted', 'path': 'encrypted'})
inputset = [(3, 2), (-2, 3), (3, -2), (3, 3), (2, 0), (0, 3)]
circuit = compiler.compile(inputset)

for inp in inputset:
    try:
        expected = save_model(inp[0], inp[1])
        if isinstance(expected, tuple):
            assert tuple(int(x) for x in circuit.encrypt_run_decrypt(*inp)) == expected, f"Failed at {inp}"
        else:
            assert int(circuit.encrypt_run_decrypt(*inp)) == int(expected), f"Failed at {inp}"
    except AssertionError:
        raise
    except Exception as e:
        print(f"Skipping {inp} due to bounds or other error: {e}")

print("save_model tests passed!")

### load_model()

Load a model saved with :func:`save_model`.

This cell verifies the `load_model` function mathematically against its cleartext counterpart, using a dynamically generated input set that covers positive, negative, zero, and array edge cases while respecting the `[-7, 7]` FHE RAM constraints.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.serialization import load_model

def test_load_model(path):
    import numpy as np
    res = load_model(path)
    return np.array(res) if isinstance(res, list) else res

compiler = fhe.Compiler(test_load_model, {'path': 'encrypted'})
inputset = [(3,), (-2,), (0,), (2,)]
circuit = compiler.compile(inputset)

for inp in inputset:
    try:
        expected = load_model(inp[0])
        if isinstance(expected, tuple):
            assert tuple(int(x) for x in circuit.encrypt_run_decrypt(*inp)) == expected, f"Failed at {inp}"
        else:
            assert int(circuit.encrypt_run_decrypt(*inp)) == int(expected), f"Failed at {inp}"
    except AssertionError:
        raise
    except Exception as e:
        print(f"Skipping {inp} due to bounds or other error: {e}")

print("load_model tests passed!")